In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
openai = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta", api_key=api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "¿Cuánto son 2 + 2?"}]
)

print(response.choices[0].message.content)

2 + 2 = 4



In [3]:
import os
import time
from typing import List, Optional
from urllib.parse import urljoin

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import WebBaseLoader
from pydantic import BaseModel, Field

# --- 1. CONFIGURACIÓN ---
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = "TU_API_KEY_AQUI"

# Headers para simular un navegador real y evitar bloqueos 403
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# --- 2. DEFINICIÓN DE DATOS (ESQUEMA) ---

class Coche(BaseModel):
    modelo: str = Field(description="El título exacto del modelo del coche.")
    precio: str = Field(description="El precio. Si no hay precio, poner 'Consultar'.")
    detalles: str = Field(description="Resumen breve de año, km y combustible.")
    
class RespuestaPagina(BaseModel):
    """
    Esta estructura le dice a la IA: 
    'Dame la lista de coches de esta página Y el enlace a la siguiente si existe'
    """
    coches: List[Coche] = Field(description="Lista de todos los coches encontrados en el listado principal.")
    siguiente_pagina: Optional[str] = Field(
        description="La URL completa o relativa que corresponde al botón 'Siguiente', 'Next' o la flecha derecha de paginación. Si no hay más páginas, devuelve null."
    )

# --- 3. INICIALIZAR GEMINI ---
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
extractor = llm.with_structured_output(RespuestaPagina)

# --- 4. FUNCIÓN DE SCRAPING RECURSIVO ---

def scrapear_autofesa_inteligente(url_inicial: str, max_paginas: int = 5):
    url_actual = url_inicial
    resultados_totales = []
    
    print(f"🚀 Iniciando scraping puro con LangChain (Max páginas: {max_paginas})")

    for i in range(max_paginas):
        print(f"\n📄 Procesando Página {i + 1}: {url_actual}")
        
        try:
            # A. CARGAR (Sin Selenium, usando Requests via LangChain)
            loader = WebBaseLoader(url_actual, header_template=HEADERS)
            docs = loader.load()
            
            # Limpieza básica para ahorrar tokens (quitamos scripts y estilos si sobran)
            contenido = docs[0].page_content
            
            # B. INFERENCIA (Gemini busca coches y el link 'siguiente')
            print("   🧠 Analizando HTML con Gemini...")
            respuesta: RespuestaPagina = extractor.invoke(contenido)
            
            # C. GUARDAR RESULTADOS
            num_encontrados = len(respuesta.coches)
            print(f"   ✅ Encontrados: {num_encontrados} coches.")
            
            for coche in respuesta.coches:
                resultados_totales.append(coche)
                # Pequeño log visual
                # print(f"      - {coche.modelo} [{coche.precio}]")

            # D. GESTIÓN DE PAGINACIÓN
            next_link = respuesta.siguiente_pagina
            
            if next_link:
                # Convertir enlace relativo (/page/2) a absoluto (https://web.com/page/2)
                url_actual = urljoin(url_actual, next_link)
                print(f"   ➡️ Siguiente página detectada: {next_link}")
                
                # Pausa de cortesía para no saturar el servidor
                time.sleep(1)
            else:
                print("   🏁 No se detectó página siguiente. Terminando.")
                break
                
        except Exception as e:
            print(f"   ❌ Error en página {i+1}: {e}")
            break

    return resultados_totales

# --- 5. EJECUCIÓN ---

if __name__ == "__main__":
    url_objetivo = "https://www.autofesa.com/coches-segunda-mano"
    
    start_time = time.time()
    datos = scrapear_autofesa_inteligente(url_objetivo, max_paginas=3) # Limitado a 3 para prueba
    end_time = time.time()
    
    print("\n" + "="*40)
    print(f"📊 RESUMEN FINAL")
    print(f"⏱️ Tiempo total: {round(end_time - start_time, 2)} segundos")
    print(f"🚗 Total coches extraídos: {len(datos)}")
    print("="*40)
    
    # Mostrar los primeros 3 ejemplos
    for i, c in enumerate(datos[:3]):
        print(f"{i+1}. {c.modelo} - {c.precio} ({c.detalles})")

🚀 Iniciando scraping puro con LangChain (Max páginas: 3)

📄 Procesando Página 1: https://www.autofesa.com/coches-segunda-mano
   🧠 Analizando HTML con Gemini...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


   ✅ Encontrados: 30 coches.
   ➡️ Siguiente página detectada: https://www.autofesa.com/coches-de-segunda-mano-ocasion/?page=2


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



📄 Procesando Página 2: https://www.autofesa.com/coches-de-segunda-mano-ocasion/?page=2
   🧠 Analizando HTML con Gemini...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


   ✅ Encontrados: 0 coches.
   🏁 No se detectó página siguiente. Terminando.

📊 RESUMEN FINAL
⏱️ Tiempo total: 19.73 segundos
🚗 Total coches extraídos: 30
1. Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY 3P # IVA DEDUCIBLE, NAVY, CUERO - 17.350 (2020, 83.900km, Gasolina)
2. Abarth 500c 1.4 595 COMPETIZIONE 180CV 2P # BUTACA DEPORTIVA, PARKTRONIC - 15.350 (2015, 97.000km, Gasolina)
3. Aixam S10 SPORT S10 SPORT - Consultar (2024, 3.274km, Diesel)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [4]:
import pandas as pd

# 1. Ejecutamos el scraper (asumiendo que ya definiste la función arriba)
url = "https://www.autofesa.com/coches-segunda-mano"
datos = scrapear_autofesa_inteligente(url, max_paginas=2)

if datos:
    # 2. Convertimos los objetos de la IA a formato compatible con Pandas
    df = pd.DataFrame([coche.model_dump() for coche in datos])

    # 3. Tu configuración exacta
    pd.set_option('display.max_rows', None)     # Mostrar todas las filas
    pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
    pd.set_option('display.width', 1000)        # Evitar cortes horizontales
    pd.set_option('display.max_colwidth', None) # Asegurar que se lea todo el texto (detalles)

    # 4. Imprimir
    print(df)
else:
    print("No se encontraron datos.")

🚀 Iniciando scraping puro con LangChain (Max páginas: 2)

📄 Procesando Página 1: https://www.autofesa.com/coches-segunda-mano
   🧠 Analizando HTML con Gemini...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


   ✅ Encontrados: 30 coches.
   ➡️ Siguiente página detectada: https://www.autofesa.com/coches-de-segunda-mano-ocasion/?page=2


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



📄 Procesando Página 2: https://www.autofesa.com/coches-de-segunda-mano-ocasion/?page=2
   🧠 Analizando HTML con Gemini...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


   ✅ Encontrados: 0 coches.
   🏁 No se detectó página siguiente. Terminando.
                                                                                                                                  modelo     precio                   detalles
0                                                         Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY 3P # IVA DEDUCIBLE, NAVY, CUERO     17.350   2020, 83.900km, Gasolina
1                                                               Abarth 500c 1.4 595 COMPETIZIONE 180CV 2P # BUTACA DEPORTIVA, PARKTRONIC     15.350   2015, 97.000km, Gasolina
2                                                                                                              Aixam S10 SPORT S10 SPORT  Consultar      2024, 3.274km, Diesel
3                                                                                            Aixam S8 COUPE S8 COUPE 9CV AUTO 3P # CUERO     10.450     2015, 44.144km, Diesel
4   Alfa Romeo 159 SPORTWAGON 2.0 JTDM SPORT PLU

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
